# 01 Data Acquisition

This notebook loads the source manufacturing dataset, builds the job-level model-ready table, calculates net cross-sectional area and shear load, and saves `data/processed_material_database.csv`. If the raw Excel workbook is not included, the notebook confirms that the processed CSV is available.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

print("Imports loaded successfully.")

from pathlib import Path
import os

# Notebook is inside notebooks/, so ../data goes to the repo data folder
data_dir = Path("../data")

excel_path = data_dir / "Materials_Database.xlsx"
processed_path = data_dir / "processed_material_database.csv"

print("Current working directory:")
print(os.getcwd())

print("\nFiles in ../data:")
print(os.listdir(data_dir))

print("\nLooking for Excel file:")
print(excel_path)
print("Excel file exists:", excel_path.exists())

print("\nLooking for processed CSV:")
print(processed_path)
print("Processed CSV exists:", processed_path.exists())


Imports loaded successfully.
Current working directory:
/Users/SenkoBaby/Desktop/15-5ph-shear-strength-ml/notebooks

Files in ../data:
['model_results.csv', 'processed_material_database.csv', 'screening_results.csv', 'model_predictions.csv', 'README.md', 'feature_list.csv', 'Materials_Database.xlsx']

Looking for Excel file:
../data/Materials_Database.xlsx
Excel file exists: True

Looking for processed CSV:
../data/processed_material_database.csv
Processed CSV exists: True


## Load source data or use processed data

The raw workbook is not required for reproducibility because the repository includes the processed CSV. If `data/Material_Database.xlsx` is available locally, this notebook will regenerate the processed CSV from the workbook.

In [6]:
if excel_path.exists():
    print("Raw workbook found. Loading Excel sheets...")

    lot = pd.read_excel(excel_path, sheet_name="LotInfo_15-5")
    shear = pd.read_excel(excel_path, sheet_name="ShearResults_15-5")
    heat_treat = pd.read_excel(excel_path, sheet_name="Heat Treat_15-5")
    summary = pd.read_excel(excel_path, sheet_name="JobSummary_15-5")

    print("Sheets loaded:")
    print(f"LotInfo_15-5:      {lot.shape}")
    print(f"ShearResults_15-5: {shear.shape}")
    print(f"Heat Treat_15-5:   {heat_treat.shape}")
    print(f"JobSummary_15-5:   {summary.shape}")

else:
    raise FileNotFoundError(
        "Raw workbook not found. Check that Materials_Database.xlsx is in the data folder."
    )

Raw workbook found. Loading Excel sheets...
Sheets loaded:
LotInfo_15-5:      (9, 25)
ShearResults_15-5: (52, 15)
Heat Treat_15-5:   (4, 8)
JobSummary_15-5:   (6, 16)


In [7]:
# Build processed dataset from job-level summary and lot information

processed_df = summary.merge(
    lot,
    on="Lot_ID",
    how="left",
    suffixes=("", "_lot")
)

# Calculate net cross-sectional area from OD and ID
if "Mean_OD_in" in processed_df.columns and "Mean_ID_in" in processed_df.columns:
    processed_df["MeanNetArea_in2"] = (
        np.pi / 4
    ) * (processed_df["Mean_OD_in"]**2 - processed_df["Mean_ID_in"]**2)

# Convert shear strength to estimated shear load
# ksi * in^2 = kips, so multiply by 1000 for lbf
if "MeanShear_ksi" in processed_df.columns and "MeanNetArea_in2" in processed_df.columns:
    processed_df["MeanShearLoad_lbf"] = (
        processed_df["MeanShear_ksi"] * processed_df["MeanNetArea_in2"] * 1000 * 2
    )

# Save processed dataset
processed_df.to_csv(processed_path, index=False)

print("Processed dataset created and saved.")
print(f"Processed dataset shape: {processed_df.shape}")
print(f"Saved to: {processed_path}")

Processed dataset created and saved.
Processed dataset shape: (6, 42)
Saved to: ../data/processed_material_database.csv


## Preview processed dataset

In [8]:
print("Processed dataset shape:")
print(processed_df.shape)

display_cols = [
    "JobNum", "Lot_ID", "PartNum", "TestStage", "AgeTemp_F", "AgeTime_hr",
    "Mean_OD_in", "Mean_ID_in", "MeanNetArea_in2",
    "MeanShear_ksi", "MeanShearLoad_lbf", "SpecimenCount"
]
display_cols = [c for c in display_cols if c in processed_df.columns]

display(processed_df[display_cols])


Processed dataset shape:
(6, 42)


,JobNum,Lot_ID,PartNum,TestStage,AgeTemp_F,AgeTime_hr,Mean_OD_in,Mean_ID_in,MeanNetArea_in2,MeanShear_ksi,MeanShearLoad_lbf,SpecimenCount
0,F2915-00,16347,115A5260-603,T1,1160,4,1.349767,0.730267,1.012049,91.038919,184271.671558,6
1,F2915-00,16347,115A5260-603,T1A,1180,4,1.349583,0.730233,1.011698,87.471372,176989.300629,6
2,F2915-00,16347,115A5260-603,T2,1180,4,1.349654,0.737531,1.003436,88.097680,176800.693623,13
3,F2916-00,16827,115A5260-603,T1,1160,4,1.349750,0.730050,1.012262,91.675099,185598.445397,6
4,F2916-00,16827,115A5260-603,T1A,1180,4,1.349650,0.721283,1.022043,87.441029,178736.972067,6
5,F2916-00,16827,115A5260-603,T2,1180,4,1.349643,0.745586,0.994030,87.320294,173597.912164,14


## Basic data quality checks

In [9]:
print("Missing values by column:")
missing = processed_df.isna().sum()
display(missing[missing > 0])

print("\nRows by lot:")
display(processed_df["Lot_ID"].value_counts())

print("\nRows by test stage:")
display(processed_df["TestStage"].value_counts())


Missing values by column:


Nb              6
AgedHard_HRC    3
GrainSize       6
dtype: int64


Rows by lot:


Lot_ID
16347    3
16827    3
Name: count, dtype: int64


Rows by test stage:


TestStage
T1     2
T1A    2
T2     2
Name: count, dtype: int64